In [ ]:
# @title ⚙️ BLOCK 1: Khởi tạo & Hàm tiện ích
import os, subprocess, sys, time, requests, json, gc, threading, re, socket, urllib.request
import ipywidgets as widgets
import zipfile, random, string
from datetime import datetime
from PIL import Image
from IPython.display import clear_output, display, HTML, Image as IPImage
from io import BytesIO

!pip install -q python-multipart

# ─── Giá trị mặc định ───────────────────────────────────────────────────────
check_value = {
    'DriveSyn'        : True,
    'Setting_Preset'  : 'None',
    'Lib'             : True,
    'FastMode'        : True,
    'root_folder'     : '/content',
    'User_folder'     : '/content/drive/MyDrive/SD-Data',
    'API_folder'      : '/content/SDVN-WebUI',
    'SDVNFolder'      : '/content/SDVN-WebUI',
    'Version'         : 'ComfyUI',
    'OptionMode'      : 'base',
    'Controlnet'      : 'none',
    'SDmodel'         : '',
    'SD15model'       : '',
    'SDXLmodel'       : '',
    'Fluxmodel'       : '',
    'SDVNmodel'       : '',
    'InpaintMd'       : '',
    'DriveLib'        : True,
    'CommandLine'     : '',
    'frontend_folder' : '/usr/local/lib/python3.12/dist-packages/comfyui_frontend_package/static',
    'Sever_Pinggy'    : '',
    # BUG FIX: 'test' chưa được khởi tạo → NameError khi chạy lần đầu
    'test'            : '',
}

if 'UI_Version' in globals():
    Version = UI_Version
for key, value in check_value.items():
    if key not in globals():
        globals()[key] = value
    elif type(globals()[key]) == str:
        globals()[key] = globals()[key].split(' ')[-1]

# ─── Hàm tiện ích ────────────────────────────────────────────────────────────
def inf(msg, style, wdth):
    btn = widgets.Button(description=msg, disabled=True,
                         button_style=style,
                         layout=widgets.Layout(min_width=wdth))
    display(btn)

def replace_text_file(file_path, old_text, new_text):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(content.replace(old_text, new_text))

def aria_down(link, path, name):
    !aria2c --console-log-level=error -c -x 8 -s 8 -k 1M {link} -d {path} -o {name}

def c_folder(path):
    os.makedirs(f'{User_folder}/{path}', exist_ok=True)

def default_file_check(folder_source, folder_check, over=False):
    flag = '-f' if over else '-n'
    for item in os.listdir(folder_source):
        src = os.path.join(folder_source, item)
        if os.path.isfile(src):
            dst = src.replace(SDVNFolder, folder_check)
            !cp {flag} {src} {dst}
        elif os.path.isdir(src):
            default_file_check(src, folder_check, over)

def civit_downlink(link):
    !wget {link} -q -O model.html
    try:
        with open('model.html', 'r', encoding='utf-8') as f:
            html_content = f.read()
        model_id = re.findall(r'"modelVersionId":(\d+),', html_content)
        if model_id:
            api_link = f'https://civitai.com/api/download/models/{model_id[0]}'
            print(f'Download model id_link: {api_link}')
            return api_link
        return 'Không tìm thấy modelVersionId.'
    except Exception as e:
        return f'Lỗi: {e}'

def check_link(link):
    if 'huggingface.co' in link:
        return link.replace('blob', 'resolve') if 'blob' in link else link
    if 'civitai.com/models' in link:
        return civit_downlink(link)
    return link

# BUG FIX: hàm token() bị định nghĩa 2 lần trong bản gốc → gộp thành 1
def token(link):
    return '?token=8c7337ac0c39fe4133ae19a3d65b806f' if 'civitai' in link else ''

def check(path):
    os.makedirs(path, exist_ok=True)

def split_string_at(link):
    delimiter = '@='
    if delimiter in link:
        parts = link.split(delimiter)
        if not any(ext in parts[1] for ext in ['.ckpt', '.gguf', '.safetensors']):
            parts[1] += '.safetensors'
        return parts[0], parts[1]
    return link, None

def download(link):
    dl = 'aria2c --console-log-level=error -c -x 8 -s 8 -k 1M'
    link = link.replace('&', '\&')
    valuelink, valuename = split_string_at(link)
    valuelink = check_link(valuelink.split('?')[0])
    if '#' not in link:
        if valuename:
            if 'civit' not in link:
                !{dl} {valuelink} -d {checkpointpatch} -o {valuename}
            else:
                !wget {valuelink}{token(valuelink)} -O {checkpointpatch}/{valuename}
        else:
            !wget {valuelink}{token(valuelink)} -P {checkpointpatch} --content-disposition

def download_controlnet(link):
    dl = 'aria2c --console-log-level=error -c -x 8 -s 8 -k 1M'
    name = link.split('/')[-1].split('?')[0]
    !{dl} {link} -d {controlnetpath} -o {name}

def download_lib(model, modellist):
    dl = 'aria2c --console-log-level=error -c -x 8 -s 8 -k 1M'
    if 'https:' in model:
        download(model)
    else:
        if not any(ext in model for ext in ['.ckpt', '.gguf', '.safetensors']):
            model += '.safetensors'
        if model in modellist:
            !{dl} {modellist[model]} -d {checkpointpatch} -o {model}

def download_txt(lines):
    for line in lines:
        download(line)

def download_txt_controlnet(txt_name):
    file_path = txt_name if '/' in txt_name else f'{controlnetpath}/{txt_name}'
    with open(file_path, 'r') as f:
        lines = [l.strip() for l in f if '#' not in l and l.strip()]
    for line in lines:
        download_controlnet(line)

def link_folder(source, target):
    !rm -rf {target}
    !ln -s {source} {target}

def run_list_txt(txt):
    with open(txt, 'r') as f:
        for line in f:
            !{line}

def defaultGraph(my_default_name, defaultGraph_name):
    if not os.path.isdir(frontend_folder):
        return
    default_path = f'{SDVNFolder}/ComfySetting/{defaultGraph_name}'
    my_default_path = f'{SDVNFolder}/ComfySetting/{my_default_name}'
    index_path = f'{frontend_folder}/index.html'
    with open(default_path, 'r', encoding='utf-8') as f:
        default = f.read()
    with open(my_default_path, 'r', encoding='utf-8') as f:
        my_default = f.read()
    with open(index_path, 'r', encoding='utf-8') as f:
        html_content = f.read()
    module_path = re.findall(r'<script type="module" crossorigin src="\./(.+?)"></script>', html_content)
    if not module_path:
        print('⚠️ Không tìm thấy module script trong index.html')
        return
    module_full = f'{frontend_folder}/{module_path[0]}'
    with open(module_full, 'r', encoding='utf-8') as f:
        content = f.read()
    with open(module_full, 'w', encoding='utf-8') as f:
        f.write(content.replace(default, my_default))

def update_huggingface_token(token_file, token_value):
    data = {}
    if os.path.exists(token_file) and os.path.getsize(token_file) > 0:
        try:
            with open(token_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError:
            pass
    data['HuggingFace'] = token_value
    with open(token_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

# ─── Hàm được Cell 10 của unified_v0_3 gọi trực tiếp ─────────────────────────
# FIX: 2 hàm này bị xoá nhầm trong bản tối ưu → Cell 10 sẽ NameError nếu thiếu

def xformers_check():
    xformersver = {
        'Forge-v2' : '0.0.27.post2',
        'Automatic': '0.0.23.post1',
        'Forge'    : '0.0.23.post1',
    }
    if Version in xformersver:
        !pip install xformers=={xformersver[Version]}
    clear_output()
    !pip show torch xformers

def install_custom():
    if Version not in ('Automatic', 'Forge', 'Forge-v2', 'ComfyUI'):
        return
    print('\033[1;32mCài đặt Extension/Node tuỳ chọn...')
    if Version != 'ComfyUI':
        %cd {Version_folder}/extensions
        exfile = ('Custom_Automatic.txt' if Version == 'Automatic'
                  else 'Custom_Forgev2.txt' if Version == 'Forge-v2'
                  else 'Custom_Forge.txt')
        setting_path = f'{User_folder}/Setting/{exfile}'
        if os.path.isfile(setting_path):
            with open(setting_path, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        !git clone {line}
        %cd /content/SDVN
    else:
        %cd {Version_folder}/custom_nodes
        node_file = f'{User_folder}/Setting/Custom_ComfyNode.txt'
        if os.path.isfile(node_file):
            with open(node_file, 'r') as f:
                for note in f:
                    note = note.strip()
                    if not note:
                        continue
                    last_segment = note.rsplit('/', 1)[-1]
                    if not os.path.exists(last_segment):
                        !git clone {note}
                        req = f'{Version_folder}/custom_nodes/{last_segment}/requirements.txt'
                        if os.path.isfile(req):
                            !pip install -q -r {req}
        %cd /content/ComfyUI
    inf('✔ Done', 'success', '150px')

print('✅ Block 1 done: Hàm tiện ích đã sẵn sàng')

In [ ]:
# @title 🌐 BLOCK 2: Tunnel Server

def _wait_port(port):
    """Chờ đến khi port được mở."""
    while True:
        time.sleep(0.5)
        s = socket.socket()
        if s.connect_ex(('127.0.0.1', port)) == 0:
            s.close(); break
        s.close()

def check_cloudflare_setting():
    domain_setting = f'{User_folder}/Setting/Domain_sever.txt'
    with open(domain_setting, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    for line in lines:
        if 'cloudflare' in line and '#' not in line:
            parts = line.split('-')
            return parts[-1].strip(), parts[1].strip()
    return None, None

def cloudflare_thread(port):
    _wait_port(port)
    flare_token, flare_domain = check_cloudflare_setting()
    if flare_token:
        subprocess.Popen(['cloudflared', 'tunnel', 'run', '--token', flare_token])
        print(f'\033[92m🔗 Link custom flare:\033[0m https://{flare_domain}')
    p = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if 'trycloudflare.com ' in l:
            print(f'\033[92m🔗 Link online:\033[0m', l[l.find('http'):], end='')
            if FastMode:
                display(HTML("<code style='color:yellow'>Đang chạy FastMode – bỏ chọn FastMode để tải đầy đủ hơn.</code>"))
            break

def pinggy_thread(port, pinggy):
    server_map = {'Auto':'','USA':'us.','Europe':'eu.','Asia':'ap.',
                  'South America':'br.','Australia':'au.'}
    sv = server_map.get(Sever_Pinggy, '')
    _wait_port(port)
    try:
        if pinggy:
            if ':' in pinggy:
                pinggy, ac, ps = pinggy.split(':')
                cmd = ['ssh','-p','443',f'-R0:localhost:{port}',
                       '-o','StrictHostKeyChecking=no','-o','ServerAliveInterval=30',
                       f'{pinggy}@{sv}pro.pinggy.io', f'"b:{ac}:{ps}"']
            else:
                cmd = ['ssh','-p','443',f'-R0:localhost:{port}',
                       '-o','StrictHostKeyChecking=no','-o','ServerAliveInterval=30',
                       f'{pinggy}@{sv}pro.pinggy.io']
        else:
            cmd = ['ssh','-p','443','-L4300:localhost:4300',
                   '-o','StrictHostKeyChecking=no','-o','ServerAliveInterval=30',
                   f'-R0:localhost:{port}','free.pinggy.io']
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in iter(p.stdout.readline, ''):
            m = re.search(r'(https?://[^\s]+)', line)
            if m:
                url = m.group(1)
                if 'dashboard.pinggy.io' in url:
                    continue
                print(f'\033[92m🔗 Link pinggy:\033[0m {url}')
                if not pinggy:
                    display(HTML("<code style='color:yellow'>Pinggy free: 60 phút. Đăng ký token tại dashboard.pinggy.io</code>"))
                break
    except Exception as e:
        print(f'❌ Pinggy lỗi: {e}')

def serveo_thread(port):
    _wait_port(port)
    try:
        p = subprocess.Popen(['ssh','-o','StrictHostKeyChecking=no',
                              '-R',f'80:localhost:{port}','serveo.net'],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in iter(p.stdout.readline, ''):
            m = re.search(r'(https?://[^\s]+)', line)
            if m:
                print(f'\033[92m🔗 Link serveo:\033[0m', m.group(1))
                break
    except Exception as e:
        print(f'❌ Serveo lỗi: {e}')

def tunnelto_thread(port, api):
    _wait_port(port)
    subprocess.run(['/root/.tunnelto/bin/tunnelto','set-auth','--key', api[0]])
    subprocess.Popen(['/root/.tunnelto/bin/tunnelto','--subdomain', api[1],'--port', str(port)])
    print(f'\033[92m🔗 Link tunnelto:\033[0m https://{api[1]}.tunn.dev')

def localtunnel_thread(port):
    _wait_port(port)
    ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
    print(f'\nLocaltunnel password: {ip}')
    p = subprocess.Popen(['lt','--port', str(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
        print(line.decode(), end='')

def tunnelmole_thread(port):
    _wait_port(port)
    try:
        p = subprocess.Popen(['tmole', str(port)],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in iter(p.stdout.readline, ''):
            m = re.search(r'(https?://[^\s]+)', line)
            if m:
                print(f'\033[92m🔗 Link tunnelmole:\033[0m', m.group(1))
                break
    except Exception as e:
        print(f'❌ Tunnelmole lỗi: {e}')

def sever_flare(port, api=None, pinggy=None):
    """Khởi tunnel song song. Chỉ start Pinggy nếu không có tunnelto."""
    if api:
        threading.Thread(target=tunnelto_thread, daemon=True, args=(port, api)).start()
    else:
        # Chỉ start Pinggy khi không có tunnelto để tránh tunnel dư thừa
        threading.Thread(target=pinggy_thread, daemon=True, args=(port, pinggy)).start()
    threading.Thread(target=cloudflare_thread, daemon=True, args=(port,)).start()

print('✅ Block 2 done: Tunnel functions sẵn sàng')

In [ ]:
# @title 📁 BLOCK 3: Tạo Folder & Cấu hình Preset

# Folder chung cho mọi version
common_folders = [
    '', 'Export', 'TrainData', 'Setting',
    'Setting/Comfy_Setting',
    'Setting/Comfy_Setting/default',
    'Setting/Comfy_Setting/default/workflows',
]

# Folder riêng theo Version
version_folders = {
    'ComfyUI': [
        'ComfyUIinput', 'Export/ComfyUI',
        'ComfyModel', 'ComfyModel/clip', 'ComfyModel/clip_vision',
        'ComfyModel/diffusion_models', 'ComfyModel/unet', 'ComfyModel/diffusers',
        'ComfyModel/upscale_models', 'ComfyModel/vae', 'ComfyModel/inpaint',
        'ComfyModel/checkpoints', 'ComfyModel/facerestore_models',
        'ComfyModel/ipadapter', 'ComfyModel/instantid', 'ComfyModel/antelopev2',
        'ComfyModel/pulid', 'ComfyModel/style_models',
        'ComfyModel/ipadapter-flux', 'ComfyModel/animatediff_models',
    ],
    'Automatic': [
        'Model', 'Lora', 'Embeddings', 'Hypernetworks',
        'wildcards', 'AnimateDiffModel', 'Lora/AnimateDiffLora',
        'Export/Automatic', 'ControlnetModel',
    ],
    'Forge': [
        'Model', 'Lora', 'Embeddings', 'Hypernetworks',
        'wildcards', 'Export/Forge', 'ControlnetModel',
    ],
    'Forge-v2': [
        'Model', 'Lora', 'Embeddings',
        'wildcards', 'Export/Forge-v2', 'ControlnetModel',
    ],
    'Kohya'      : ['TrainData'],
    'Fooocus'    : ['Export/Fooocus', 'Lora'],
    'AutoRetouch': ['Export/AutoRetouch'],
    'FluxGym'    : ['TrainData'],
}

%cd {root_folder}

# Tạo folder chung
for folder in common_folders:
    c_folder(folder)

# Tạo folder theo version hiện tại (không tạo 30+ folder không cần)
for folder in version_folders.get(Version, []):
    c_folder(folder)

default_file_check(f'{SDVNFolder}/Setting', User_folder)

# Update path trong WebUI_Setting.json
file_path = f'{User_folder}/Setting/WebUI_Setting.json'
replace_text_file(file_path, '/content/drive/MyDrive/SD-Data', User_folder)

clear_output()

# ─── Preset ──────────────────────────────────────────────────────────────────
Preset = {
    'Comfy_SD_fast(2min)'        : [True, 'ComfyUI',    'base',   'base', 'RealisticVision51',  'sd_defaultGraph.js'],
    'Comfy_flux_fast(3min)'      : [True, 'ComfyUI',    'base',   'base', 'Flux_dev_v1-fp8',    'flux_defaultGraph.js'],
    'Comfy_flux_base(7min)'      : [False,'ComfyUI',    'base',   'base', 'Flux_dev_v1-fp8',    'flux_defaultGraph.js'],
    'Comfy_flux_full(9min)'      : [False,'ComfyUI',    'full',   'base', 'Flux_dev_v1-fp8',    'flux_defaultGraph.js'],
    'Comfy_flux_photo(12min)'    : [False,'ComfyUI',    '+photo', 'base', 'Flux_dev_v1-fp8',    'flux_defaultGraph.js'],
    'Forge_flux_fast(5min)'      : [True, 'Forge-v2',   'base',   'base', 'Flux_dev_v1-nf4',    'flux_defaultGraph.js'],
    'Forge_flux_base(9min)'      : [False,'Forge-v2',   'base',   'base', 'Flux_dev_v1-nf4',    'flux_defaultGraph.js'],
    'Automatic_SD_fast(5min)'    : [True, 'Automatic',  'base',   'base', 'RealisticVision51',  'sd_defaultGraph.js'],
    'Automatic_SD_base(9min)'    : [False,'Automatic',  'base',   'base', '',                   'sd_defaultGraph.js'],
    'Automatic_SD_full(11min)'   : [False,'Automatic',  'full',   'base', '',                   'sd_defaultGraph.js'],
    'Automatic_SD_photo(12min)'  : [False,'Automatic',  '+photo', 'base', '',                   'sd_defaultGraph.js'],
    'ComfyWf_Auto_SwapFace'      : [False,'ComfyUI',    'base',   'none', 'AdamXL-v3',          'Auto_SwapFace_Graph.js'],
}

preset_keys = ['FastMode','Version','OptionMode','Controlnet','SDmodel','my_default_name']
if Setting_Preset != 'None' and Setting_Preset in Preset:
    for i, key in enumerate(preset_keys):
        globals()[key] = Preset[Setting_Preset][i]

# FastMode override
if FastMode:
    Controlnet = 'none'
    Lib = False

print(f'✅ Block 3 done | Version={Version} | FastMode={FastMode} | OptionMode={OptionMode}')

In [ ]:
# @title 📦 BLOCK 4: Cài đặt UI & Download Controlnet

display(HTML("<h2 style='color:lightgreen'>⏳ Đang cài đặt... vui lòng đợi 4-5 phút</h2>"))
image_url = 'https://github.com/StableDiffusionVN/SDVN-WebUI/blob/main/huongdan.jpg?raw=true'
try:
    resp = requests.get(image_url, timeout=10)
    display(IPImage(data=resp.content))
except Exception:
    pass

# ─── Controlnet ──────────────────────────────────────────────────────────────
is_sd_version = Version in ('Automatic','Forge','Forge-v2','ComfyUI')
if is_sd_version:
    controlnetpath = f'{SDVNFolder}/ControlNet'
    list_controlnet = []
    if 'https' in Controlnet:
        list_controlnet += Controlnet.split(',')
        Controlnet = 'none'
    if OptionMode == '+mhd':
        list_controlnet.append('https://huggingface.co/StableDiffusionVN/XLControlnet/resolve/main/xinsir/xinsir_union_promax-sdxl.safetensors')
    if Setting_Preset == 'ComfyWf_Auto_SwapFace':
        list_controlnet.append('https://huggingface.co/StableDiffusionVN/Basecontrolnet/resolve/main/InstantID/control_instant_id_sdxl.safetensors')
    for lnk in list_controlnet:
        download_controlnet(lnk)

    if Controlnet != 'none':
        download_txt_controlnet('base.txt' if OptionMode == 'base' else 'base_full.txt')
    for cn_key, cn_file in [('+SD15','+SD15.txt'),('+SDXL','+SDXL.txt'),('+Flux','+Flux.txt')]:
        if Controlnet == cn_key:
            download_txt_controlnet(cn_file)
    if Controlnet == 'User_list':
        download_txt_controlnet(f'{User_folder}/Setting/Custom_User_Controlnet_list.txt')
    link_folder(f'{User_folder}/ControlnetModel', f'{controlnetpath}/ControlnetModel')

    # Library (wildcards, embeddings, lora)
    if Lib:
        !git clone https://github.com/StableDiffusionVN/WC-SDVN
        link_folder(f'{root_folder}/WC-SDVN', f'{User_folder}/wildcards/WC-SDVN')
        !git clone https://huggingface.co/StableDiffusionVN/clone
        link_folder(f'{root_folder}/clone/Negative', f'{User_folder}/Embeddings/Negative')
        link_folder(f'{root_folder}/clone/Lora', f'{User_folder}/Lora/Lora')

# ─── Cài đặt từng UI ─────────────────────────────────────────────────────────
# BUG FIX: bỏ capture_output để lỗi cài đặt hiện ra được

if Version in ('Forge','Automatic','Forge-v2'):
    !pip install -q -U wandb==0.16.0
    Version_folder = f'{root_folder}/SDVN'

    # BUG FIX: 'test' đã được khởi tạo trong check_value nên không còn NameError
    if test != Version:
        !rm -rf /content/SDVN
    test = Version  # noqa: F841  (cập nhật global)

    if Version == 'Automatic':
        !git clone https://github.com/phamhungd/SDVN
    elif Version == 'Forge':
        !git clone -b F-19 https://github.com/phamhungd/SDVN-Forge /content/SDVN
    elif Version == 'Forge-v2':
        !git clone https://github.com/lllyasviel/stable-diffusion-webui-forge /content/SDVN

    Auto_ex = 'SDVN-WebUI/Extension/Automatic'
    if Version != 'Forge-v2':
        if FastMode:
            run_list_txt(f'{Auto_ex}/ex_fast.txt')
        else:
            run_list_txt(f"{Auto_ex}/{'ex_base.txt' if OptionMode == 'base' else 'ex_full.txt'}")
            extra_map = {'+photo':'ex_photo.txt','+video':'ex_video.txt','+dev':'ex_dev.txt'}
            if OptionMode in extra_map:
                run_list_txt(f'{Auto_ex}/{extra_map[OptionMode]}')
    elif not FastMode:
        run_list_txt(f'{SDVNFolder}/Extension/Forgev2/forge_ex.txt')

    link_folder(controlnetpath, f'{Version_folder}/models/ControlNet')
    aria_down('https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.ckpt',
              f'{Version_folder}/models/VAE', 'VAE84.vae.pt')
    aria_down('https://huggingface.co/phamhungd/GuoZovya/resolve/main/4x-UltraSharp.ckpt',
              f'{Version_folder}/models/ESRGAN', '4x-UltraSharp.pth')
    link_folder(f'{User_folder}/wildcards', f'{Version_folder}/extensions/Dynamic-prompts/wildcards')
    default_file_check(f'{SDVNFolder}/SDVN', root_folder, True)

elif Version == 'ComfyUI':
    current_date = datetime.now().date()
    os.makedirs(f'{User_folder}/Export/ComfyUI/{current_date}', exist_ok=True)
    %cd {root_folder}
    Version_folder = f'{root_folder}/ComfyUI'
    !git clone https://github.com/Comfy-Org/ComfyUI

    if 'Comfy_commit' in globals() and Comfy_commit and Comfy_commit != 'None':
        %cd {Version_folder}
        !git checkout {Comfy_commit}

    !sed -i '1s|.*|comfyui-frontend-package==1.39.19|' /content/ComfyUI/requirements.txt
    !pip install -q -r /content/ComfyUI/requirements.txt
    default_file_check(f'{SDVNFolder}/templates', frontend_folder, True)

    %cd {Version_folder}/custom_nodes
    comfy_node = f'{SDVNFolder}/Extension/Comfy'
    if FastMode:
        run_list_txt(f'{comfy_node}/node_fast.txt')
    else:
        run_list_txt(f"{comfy_node}/{'node_base.txt' if OptionMode == 'base' else 'node_full.txt'}")
        extra_map = {'+photo':'node_photo.txt','+video':'node_video.txt',
                     '+train':'node_dev.txt','+mhd':'node_mhd.txt'}
        if OptionMode in extra_map:
            run_list_txt(f'{comfy_node}/{extra_map[OptionMode]}')

elif Version == 'Kohya':
    Version_folder = f'{root_folder}/KohyaUI'
    %cd {root_folder}
    !git clone -b v24.1.4 https://github.com/phamhungd/KohyaUI
    %cd {Version_folder}
    !./setup.sh

elif Version == 'Fooocus':
    Version_folder = f'{root_folder}/Fooocus'
    %cd {root_folder}
    !git clone https://github.com/lllyasviel/Fooocus
    %cd {Version_folder}
    link_folder(f'{User_folder}/Export/Fooocus', f'{Version_folder}/outputs')
    link_folder(f'{User_folder}/Lora', f'{Version_folder}/models/loras')
    !cp -f {SDVNFolder}/FooocusSetting/Fooocus_config.txt {Version_folder}/config.json

elif Version == 'AutoRetouch':
    Version_folder = f'{root_folder}/AutoRetouch'
    !git clone https://github.com/phamhungd/auto_retoucher {Version_folder}
    !pip install -q -r {Version_folder}/requirements.txt
    link_folder(f'{User_folder}/Export/AutoRetouch', f'{Version_folder}/outputs')

elif Version == 'FluxGym':
    Version_folder = f'{root_folder}/fluxgym'
    %cd /content
    !git clone https://github.com/cocktailpeanut/fluxgym
    !git clone -b sd3 https://github.com/kohya-ss/sd-scripts {Version_folder}/sd-scripts
    %cd {Version_folder}/sd-scripts
    !pip install -q -r requirements.txt
    %cd {Version_folder}
    !pip install -q -r requirements.txt

inf('✔ Cài đặt xong', 'success', '150px')

In [ ]:
# @title 🔗 BLOCK 5: Download Model & Cấu hình ComfyUI
#
# ⚠️  LƯU Ý KHI DÙNG VỚI unified_v0_3_SDVN_ComfyUI.ipynb:
#
#  Cell B (unified) sẽ chạy SAU block này và sẽ:
#    - rm -rf toàn bộ /content/ComfyUI/models/* rồi tạo symlink file từ Drive
#    - ln -s Drive/ComfyUI_Backup/custom_nodes → /content/ComfyUI/custom_nodes
#
#  Hệ quả:
#    1. link_folder(checkpointpatch, Version_path)  → bị Cell B xoá → BỎ
#    2. Symlink config files vào custom_nodes/*     → bị Cell B xoá → BỎ
#    3. Model download vào checkpointpatch          → bị orphan    → BỎ fallback FastMode
#       (models nên đặt trên Drive, Cell B tự link)
#
#  Nếu bạn chạy Script này STANDALONE (không có unified notebook):
#    → Bỏ comment 3 dòng được đánh dấu [STANDALONE ONLY] bên dưới

# ─── Download model (nếu được chỉ định qua biến, không phải từ Drive) ────────
checkpointpatch = f'{root_folder}/Checkpoint'
os.makedirs(checkpointpatch, exist_ok=True)

with open(f'{SDVNFolder}/model_lib.json', 'r') as f:
    modellist = json.load(f)

# [STANDALONE ONLY] Bỏ comment dòng dưới nếu chạy không có unified notebook:
# if FastMode and not SDmodel:
#     SDmodel = 'https://huggingface.co/StableDiffusionVN/Flux2/blob/main/Checkpoint/Flux2-klein-9b.safetensors@=Flux2-klein-9b'

for model_var in ['SDmodel','SD15model','SDXLmodel','Fluxmodel','SDVNmodel','InpaintMd']:
    val = globals().get(model_var, '')
    if val:
        download_lib(val, modellist)

if not FastMode and 'ComfyWf' not in Setting_Preset and Version not in ('FluxGym',):
    custom_list = f'{User_folder}/Setting/Custom_Model_List.txt'
    if os.path.isfile(custom_list):
        with open(custom_list, 'r') as f:
            custom_lines = [l.strip() for l in f if l.strip()]
        download_txt(custom_lines)

# ─── Symlink cho version KHÔNG phải ComfyUI ──────────────────────────────────
# (ComfyUI: Cell B của unified notebook xử lý toàn bộ symlink)
if Version != 'ComfyUI':
    checkpoint_path_map = {
        'Automatic'  : f'{Version_folder}/models/Stable-diffusion/Checkpoint',
        'Forge'      : f'{Version_folder}/models/Stable-diffusion/Checkpoint',
        'Forge-v2'   : f'{Version_folder}/models/Stable-diffusion/Checkpoint',
        'Fooocus'    : f'{Version_folder}/models/checkpoints',
        'AutoRetouch': f'{Version_folder}/models',
    }
    Version_path = checkpoint_path_map.get(Version, '')
    if Version_path and Version not in ('Kohya', 'FluxGym'):
        link_folder(checkpointpatch, Version_path)  # [STANDALONE ONLY] an toàn cho non-ComfyUI

# ─── Cấu hình ComfyUI (chỉ phần KHÔNG bị Cell B override) ───────────────────
if Version == 'ComfyUI':
    current_date = datetime.now().date()
    %cd {root_folder}

    # defaultGraph: cần chạy trước Cell B vì Cell B không đụng frontend_folder
    if Setting_Preset == 'None':
        my_default_name = globals().get('my_default_name', 'sd_defaultGraph.js')
    defaultGraph(my_default_name, 'defaultGraph.js')

    # extra_model_paths.yaml: nằm trong Version_folder (không bị Cell B xoá)
    !cp -f {SDVNFolder}/ComfySetting/extra_model_paths.yaml {Version_folder}
    replace_text_file(f'{Version_folder}/extra_model_paths.yaml',
                      '/content/drive/MyDrive/SD-Data', User_folder)

    # Symlink output folder (nằm trong Version_folder, Cell B không đụng)
    os.makedirs(f'{User_folder}/Export/ComfyUI/{current_date}', exist_ok=True)
    link_folder(f'{User_folder}/Export/ComfyUI/{current_date}',
                f'{Version_folder}/output')

    # Favicon (nằm trong custom_nodes → Cell B sẽ override, chạy lại sau Cell B nếu cần)
    # !cp -f {SDVNFolder}/ComfySetting/favicon.ico {Version_folder}/custom_nodes/ComfyUI-Custom-Scripts/web/js/assets
    # !cp -f {SDVNFolder}/ComfySetting/favicon-active.ico {Version_folder}/custom_nodes/ComfyUI-Custom-Scripts/web/js/assets

clear_output()
inf('✔ Hoàn tất cài đặt – Sẵn sàng chạy!', 'success', '250px')

In [ ]:
# @title ▶️ BLOCK 6: Định nghĩa hàm chạy UI
#
# Block này CHỈ định nghĩa các hàm — KHÔNG gọi run_ver().
# run_ver() được gọi bởi Cell 10 của unified_v0_3_SDVN_ComfyUI.ipynb
# (sau khi Cell A/B/C đã setup xong dependency, Drive symlink, và torch patch).
#
# Nếu bạn dùng Script này STANDALONE (không qua unified notebook):
#   → Bỏ comment dòng cuối cùng: run_ver(Version=Version, CommandLine=CommandLine)

def run_comfy_background(final_arg):
    !pkill -f 'main.py'
    os.system(f'python main.py {final_arg} &')

def install_app(appfolder):
    !pip install -q -r {appfolder}/requirements.txt
    !cp -rf {appfolder}/example {User_folder}/ComfyUIinput
    default_wf(appfolder)
    %cd {Version_folder}/custom_nodes
    with open(f'{appfolder}/node.txt', 'r') as f:
        notes = f.readlines()
    for note in notes:
        last_segment = note.rsplit('/', 1)[-1].strip()
        if not os.path.exists(last_segment):
            !git clone {note}
            req = f'{Version_folder}/custom_nodes/{last_segment}/requirements.txt'
            if os.path.isfile(req):
                !pip install -q -r {req}
            !ln -s {appfolder}/prompt.dat /content/ComfyUI/prompt.dat
            !ln -s {appfolder}/workflow.json /content/ComfyUI/styles.csv
    %cd /content/ComfyUI

def default_wf(appfolder):
    with open(f'{appfolder}/workflow.json', 'r', encoding='utf-8') as f:
        json_data = json.load(f)
    js_content = f'const defaultGraph = {json.dumps(json_data, indent=2)};'
    with open(f'{SDVNFolder}/ComfySetting/workflow.js', 'w', encoding='utf-8') as f:
        f.write(js_content)
    defaultGraph('workflow.js', 'sd_defaultGraph.js')

def random_app(parent_folder):
    subfolders = [f for f in os.listdir(parent_folder)
                  if os.path.isdir(os.path.join(parent_folder, f))]
    if not subfolders:
        return None
    folder_main = os.path.join(parent_folder, random.choice(subfolders))
    folder_name = ''.join(random.choices(string.ascii_letters, k=5))
    r = os.path.join(folder_main, folder_name)
    os.makedirs(r, exist_ok=True)
    return r

def extract_app(App_name, Ver):
    global API_folder
    API_folder = random_app('/content/ComfyUI/comfy')
    host = 'http://stablediffusion.vn/wp-content/uploads/'
    dl_link = f'{host}/store/{App_name}/{Ver}.zip'
    !wget -q {dl_link} -O {API_folder}/{Ver}.zip
    with zipfile.ZipFile(f'{API_folder}/{Ver}.zip', 'r') as z:
        z.extractall(API_folder)
    os.remove(f'{API_folder}/{Ver}.zip')
    install_app(API_folder)

def run_ver(Version='ComfyUI', CommandLine='', tunnelto=None, Run='None', pinggy=None):
    %cd {Version_folder}

    if Run != 'None' or 'App_name' in globals():
        CommandLine = '--disable-metadata'

    if tunnelto is None and pinggy is None:
        domain_setting = f'{User_folder}/Setting/Domain_sever.txt'
        with open(domain_setting, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        for line in lines:
            if 'tunn.dev' in line and '#' not in line:
                tunnelto = [line.split('-')[1].strip(),
                            line.split('-')[0].split('.tunn.dev')[0].strip()]
            if 'pinggy' in line and '#' not in line:
                pinggy = line.split('-')[1].strip()

    if Version in ('Automatic', 'Forge', 'Forge-v2'):
        sever_flare(7860, tunnelto, pinggy)
        drive_arg = (f'--ckpt-dir {User_folder}/Model --embeddings-dir {User_folder}/Embeddings'
                     if DriveLib else '')
        final_arg = (f"{'--xformers' if not FastMode else ''} --theme dark "
                     f'--enable-insecure-extension-access --disable-safe-unpickle '
                     f'--share --no-half-vae {CommandLine} '
                     f'--ui-settings-file {User_folder}/Setting/WebUI_Setting.json '
                     f'--styles-file {User_folder}/Setting/WebUI_Styles.csv '
                     f'--lora-dir {User_folder}/Lora {drive_arg}')
        !python launch.py {final_arg}

    elif Version == 'ComfyUI':
        sever_flare(8888, tunnelto, pinggy)
        final_arg = (f'--preview-method auto --port 8888 '
                     f'--input-directory {User_folder}/ComfyUIinput {CommandLine}')
        if Run == 'None':
            if 'App_name' in globals():
                if API_folder == '/content/SDVN-WebUI':
                    extract_app(App_name, Ver)
                    clear_output()
                run_comfy_background(final_arg)
                !python {API_folder}/app.py --share
            else:
                !python main.py {final_arg}
        else:
            run_comfy_background(final_arg)
            !python {API_folder}/API/{Run}/app.py --share

    elif Version == 'Kohya':
        display(HTML("<h3 style='color:lightgreen'>"
                     "<code>cd KohyaUI; python kohya_gui.py --share --headless</code></h3>"))
        !python {Version_folder}/kohya_gui.py --share --headless

    elif Version == 'Fooocus':
        !python {Version_folder}/entry_with_update.py --share --theme dark \
                --disable-preset-download {CommandLine}

    elif Version == 'AutoRetouch':
        !python {Version_folder}/auto_retoucher.py

    elif Version == 'FluxGym':
        %run {SDVNFolder}/Run_Fluxgym.ipynb

print('✅ Block 6 done: Tất cả hàm đã định nghĩa – chờ Cell 10 gọi run_ver()')

# [STANDALONE ONLY] Bỏ comment dòng dưới nếu chạy không qua unified notebook:
# run_ver(Version=Version, CommandLine=CommandLine)